# Volatility Surface & Options Greeks Analysis

**CQF-Level Example**: Build volatility surfaces and analyze options Greeks:
- VIX term structure analysis
- Options chain fetching and IV extraction
- Volatility smile/skew visualization
- Greeks calculation (Delta, Gamma, Vega, Theta)
- Surface fitting and arbitrage detection

**Connectors Used:**
- `qj.cboe` - VIX data, options chains
- `qj.eod` - Underlying prices

**API:** https://api.quantjourney.cloud

## Run Output

![21_volatility_surface_greeks](../plots/21_volatility_surface_greeks_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"

from scipy.stats import norm
from scipy.interpolate import griddata
from datetime import datetime, timedelta

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")
qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Black-Scholes Greeks Functions

In [ ]:
def bs_d1_d2(S, K, T, r, sigma):
    """Calculate d1 and d2 for Black-Scholes."""
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return d1, d2

def bs_price(S, K, T, r, sigma, option_type='call'):
    """Black-Scholes option price."""
    d1, d2 = bs_d1_d2(S, K, T, r, sigma)
    if option_type == 'call':
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else:
        return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

def bs_delta(S, K, T, r, sigma, option_type='call'):
    d1, _ = bs_d1_d2(S, K, T, r, sigma)
    return norm.cdf(d1) if option_type == 'call' else norm.cdf(d1) - 1

def bs_gamma(S, K, T, r, sigma):
    d1, _ = bs_d1_d2(S, K, T, r, sigma)
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def bs_vega(S, K, T, r, sigma):
    d1, _ = bs_d1_d2(S, K, T, r, sigma)
    return S * norm.pdf(d1) * np.sqrt(T) / 100  # per 1% vol change

def bs_theta(S, K, T, r, sigma, option_type='call'):
    d1, d2 = bs_d1_d2(S, K, T, r, sigma)
    theta1 = -S*norm.pdf(d1)*sigma / (2*np.sqrt(T))
    if option_type == 'call':
        theta2 = -r*K*np.exp(-r*T)*norm.cdf(d2)
    else:
        theta2 = r*K*np.exp(-r*T)*norm.cdf(-d2)
    return (theta1 + theta2) / 365  # daily theta

def implied_vol(price, S, K, T, r, option_type='call', tol=1e-6, max_iter=100):
    """Newton-Raphson implied volatility calculation."""
    sigma = 0.3  # initial guess
    for _ in range(max_iter):
        bs_price_val = bs_price(S, K, T, r, sigma, option_type)
        vega_val = bs_vega(S, K, T, r, sigma) * 100
        if vega_val < 1e-8:
            break
        diff = bs_price_val - price
        if abs(diff) < tol:
            return sigma
        sigma = sigma - diff / vega_val
        sigma = max(0.01, min(sigma, 5.0))  # bounds
    return sigma

print("✓ Greeks functions defined")


## 2. Fetch VIX Term Structure

In [ ]:
# Get VIX term structure
vix_term = qj.cboe.get_vix_term_structure()
vix_data = vix_term.get('value', vix_term) if isinstance(vix_term, dict) else vix_term

if isinstance(vix_data, list) and len(vix_data) > 0:
    vix_df = pd.DataFrame(vix_data)
    print(f"VIX Term Structure Points: {len(vix_df)}")
    print(vix_df.head(10))
else:
    # Simulate term structure for demo
    print("Using simulated VIX term structure")
    vix_df = pd.DataFrame({
        'days_to_expiry': [7, 14, 30, 60, 90, 120, 180, 270, 365],
        'vix': [18.5, 19.2, 20.1, 21.3, 22.0, 22.5, 23.1, 23.5, 24.0]
    })


In [ ]:
# Historical VIX data
vix_hist = qj.cboe.get_vix_data(start_date="2020-01-01", end_date="2024-12-31")
vix_hist_data = vix_hist.get('value', vix_hist) if isinstance(vix_hist, dict) else vix_hist

if isinstance(vix_hist_data, list) and len(vix_hist_data) > 0:
    vix_hist_df = pd.DataFrame(vix_hist_data)
    vix_hist_df['date'] = pd.to_datetime(vix_hist_df['date'])
    vix_hist_df = vix_hist_df.set_index('date')
    print(f"Historical VIX: {len(vix_hist_df)} days")
else:
    print("VIX data not available, skipping historical analysis")
    vix_hist_df = None


In [ ]:
# VIX regime analysis (if data available)
if vix_hist_df is not None:
    vix_col = 'close' if 'close' in vix_hist_df.columns else vix_hist_df.columns[0]
    vix_hist_df['vix'] = pd.to_numeric(vix_hist_df[vix_col], errors='coerce')
    vix_hist_df = vix_hist_df.dropna(subset=['vix'])
    
    # VIX regimes
    vix_hist_df['regime'] = pd.cut(
        vix_hist_df['vix'],
        bins=[0, 15, 20, 25, 30, 100],
        labels=['Very Low', 'Low', 'Normal', 'Elevated', 'High']
    )
    
    print("\nVIX Regime Distribution:")
    print(vix_hist_df['regime'].value_counts())
    
    # Plot VIX history
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=vix_hist_df.index, y=vix_hist_df['vix'],
        mode='lines', name='VIX',
        line=dict(color='orange')
    ))
    fig.add_hline(y=20, line_dash="dash", line_color="yellow", annotation_text="Normal")
    fig.add_hline(y=30, line_dash="dash", line_color="red", annotation_text="Elevated")
    
    fig.update_layout(
        title='VIX Historical Levels',
        template='plotly_dark',
        height=400
    )
    fig.show()


## 3. Fetch Options Chain

In [ ]:
# Get SPY underlying quote
# Note: CBOE API uses 'symbol' parameter for quotes
symbol = 'SPY'
quote = qj.cboe.get_underlying_quote(symbol=symbol)
quote_data = quote.get('value', quote) if isinstance(quote, dict) else quote

if quote_data:
    if isinstance(quote_data, dict):
        spot = float(quote_data.get('last', quote_data.get('close', quote_data.get('price', 470))))
    else:
        spot = 470  # fallback
else:
    spot = 470  # fallback
print(f"SPY Spot Price: ${spot:.2f}")


In [ ]:
# Get options expirations
try:
    expirations = qj.cboe.get_options_expirations(symbol=symbol)
    exp_data = expirations.get('value', expirations) if isinstance(expirations, dict) else expirations
    
    # Handle different response formats
    if isinstance(exp_data, dict) and 'expirations' in exp_data:
        exp_data = exp_data['expirations']
    elif isinstance(exp_data, dict) and 'data' in exp_data:
        exp_data = exp_data['data']
    
    if isinstance(exp_data, list) and len(exp_data) > 0:
        print(f"Available expirations: {len(exp_data)}")
        print(exp_data[:10])
        # Extract date string if nested dict
        if isinstance(exp_data[0], dict):
            exp_dates = [e.get('expiration', e.get('date', str(e))) for e in exp_data]
        else:
            exp_dates = exp_data
        expiry = exp_dates[2] if len(exp_dates) > 2 else exp_dates[0]
    else:
        print("No expirations found, using fallback")
        expiry = (datetime.now() + timedelta(days=30)).strftime('%Y-%m-%d')
except Exception as e:
    print(f"Error fetching expirations: {e}")
    expiry = (datetime.now() + timedelta(days=30)).strftime('%Y-%m-%d')

print(f"\nSelected expiry: {expiry}")


In [ ]:
# Get options chain
chain = qj.cboe.get_options_chain(symbol=symbol, expiration_date=expiry)
chain_data = chain.get('value', chain) if isinstance(chain, dict) else chain

if chain_data and isinstance(chain_data, list) and len(chain_data) > 0:
    options_df = pd.DataFrame(chain_data)
    print(f"Options chain: {len(options_df)} contracts")
    print(options_df.head())
else:
    # Generate synthetic options for demo (CBOE API may not have live data)
    print("Generating synthetic options chain for demo...")
    strikes = np.arange(spot * 0.85, spot * 1.15, 5)
    r = 0.05
    T = 30 / 365
    
    options_list = []
    for K in strikes:
        moneyness = K / spot
        # Volatility smile
        iv = 0.18 + 0.15 * (moneyness - 1)**2 + 0.05 * max(0, 1 - moneyness)
        
        for opt_type in ['call', 'put']:
            price = bs_price(spot, K, T, r, iv, opt_type)
            options_list.append({
                'strike': K,
                'type': opt_type,
                'bid': price * 0.98,
                'ask': price * 1.02,
                'mid': price,
                'iv': iv,
                'volume': np.random.randint(100, 5000),
                'open_interest': np.random.randint(1000, 50000)
            })
    options_df = pd.DataFrame(options_list)
    print(f"Synthetic options: {len(options_df)} contracts")


## 4. Volatility Smile Analysis

In [ ]:
# Calculate implied volatility if not present
if 'iv' not in options_df.columns:
    r = 0.05
    T = 30 / 365  # approximate
    ivs = []
    for _, row in options_df.iterrows():
        price = row.get('mid', (row.get('bid', 0) + row.get('ask', 0)) / 2)
        if price > 0:
            iv = implied_vol(price, spot, row['strike'], T, r, row['type'])
            ivs.append(iv)
        else:
            ivs.append(np.nan)
    options_df['iv'] = ivs

# Separate calls and puts
calls = options_df[options_df['type'] == 'call'].copy()
puts = options_df[options_df['type'] == 'put'].copy()

calls['moneyness'] = calls['strike'] / spot
puts['moneyness'] = puts['strike'] / spot


In [ ]:
# Volatility smile plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=calls['moneyness'], y=calls['iv'] * 100,
    mode='markers+lines', name='Calls',
    marker=dict(color='green', size=8)
))

fig.add_trace(go.Scatter(
    x=puts['moneyness'], y=puts['iv'] * 100,
    mode='markers+lines', name='Puts',
    marker=dict(color='red', size=8)
))

fig.add_vline(x=1.0, line_dash="dash", line_color="white", annotation_text="ATM")

fig.update_layout(
    title=f'{symbol} Volatility Smile (Expiry: {expiry})',
    xaxis_title='Moneyness (K/S)',
    yaxis_title='Implied Volatility (%)',
    template='plotly_dark',
    height=500
)
fig.show()

# Skew metrics
atm_mask = (calls['moneyness'] > 0.98) & (calls['moneyness'] < 1.02)
otm_put_mask = (puts['moneyness'] > 0.90) & (puts['moneyness'] < 0.95)

atm_iv = calls.loc[atm_mask, 'iv'].mean() if atm_mask.any() else np.nan
otm_put_iv = puts.loc[otm_put_mask, 'iv'].mean() if otm_put_mask.any() else np.nan
skew = (otm_put_iv - atm_iv) * 100 if not np.isnan(atm_iv) else np.nan

print(f"\nVolatility Skew Metrics:")
print(f"  ATM IV: {atm_iv*100:.1f}%" if not np.isnan(atm_iv) else "  ATM IV: N/A")
print(f"  OTM Put IV (90-95%): {otm_put_iv*100:.1f}%" if not np.isnan(otm_put_iv) else "  OTM Put IV: N/A")
print(f"  Skew (OTM Put - ATM): {skew:.1f}%" if not np.isnan(skew) else "  Skew: N/A")


## 5. Greeks Surface

In [ ]:
# Calculate Greeks for all options
r = 0.05
T = 30 / 365

calls['delta'] = calls.apply(lambda x: bs_delta(spot, x['strike'], T, r, x['iv'], 'call'), axis=1)
calls['gamma'] = calls.apply(lambda x: bs_gamma(spot, x['strike'], T, r, x['iv']), axis=1)
calls['vega'] = calls.apply(lambda x: bs_vega(spot, x['strike'], T, r, x['iv']), axis=1)
calls['theta'] = calls.apply(lambda x: bs_theta(spot, x['strike'], T, r, x['iv'], 'call'), axis=1)

puts['delta'] = puts.apply(lambda x: bs_delta(spot, x['strike'], T, r, x['iv'], 'put'), axis=1)
puts['gamma'] = puts.apply(lambda x: bs_gamma(spot, x['strike'], T, r, x['iv']), axis=1)
puts['vega'] = puts.apply(lambda x: bs_vega(spot, x['strike'], T, r, x['iv']), axis=1)
puts['theta'] = puts.apply(lambda x: bs_theta(spot, x['strike'], T, r, x['iv'], 'put'), axis=1)


In [ ]:
# Greeks visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Delta', 'Gamma', 'Vega', 'Theta']
)

# Delta
fig.add_trace(go.Scatter(x=calls['strike'], y=calls['delta'], name='Call Δ', line=dict(color='green')), row=1, col=1)
fig.add_trace(go.Scatter(x=puts['strike'], y=puts['delta'], name='Put Δ', line=dict(color='red')), row=1, col=1)

# Gamma
fig.add_trace(go.Scatter(x=calls['strike'], y=calls['gamma'], name='Call Γ', line=dict(color='green')), row=1, col=2)
fig.add_trace(go.Scatter(x=puts['strike'], y=puts['gamma'], name='Put Γ', line=dict(color='red')), row=1, col=2)

# Vega
fig.add_trace(go.Scatter(x=calls['strike'], y=calls['vega'], name='Call ν', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=puts['strike'], y=puts['vega'], name='Put ν', line=dict(color='red')), row=2, col=1)

# Theta
fig.add_trace(go.Scatter(x=calls['strike'], y=calls['theta'], name='Call θ', line=dict(color='green')), row=2, col=2)
fig.add_trace(go.Scatter(x=puts['strike'], y=puts['theta'], name='Put θ', line=dict(color='red')), row=2, col=2)

fig.update_layout(
    title=f'{symbol} Options Greeks by Strike',
    template='plotly_dark',
    height=600,
    showlegend=False
)
fig.show()


## 6. Volatility Surface (Multiple Expiries)

In [ ]:
# Generate synthetic vol surface for multiple expiries
expiries_days = [7, 14, 30, 60, 90, 180]
strikes = np.arange(spot * 0.85, spot * 1.15, 2.5)

vol_surface = []
for T_days in expiries_days:
    T = T_days / 365
    for K in strikes:
        moneyness = K / spot
        # Term structure + smile
        base_vol = 0.16 + 0.02 * np.sqrt(T_days / 30)  # term structure
        smile = 0.12 * (moneyness - 1)**2 + 0.04 * max(0, 1 - moneyness)  # skew
        iv = base_vol + smile
        
        vol_surface.append({
            'strike': K,
            'days': T_days,
            'moneyness': moneyness,
            'iv': iv
        })

vol_df = pd.DataFrame(vol_surface)


In [ ]:
# 3D Volatility Surface
pivot = vol_df.pivot(index='days', columns='moneyness', values='iv')

fig = go.Figure(data=[go.Surface(
    z=pivot.values * 100,
    x=pivot.columns,
    y=pivot.index,
    colorscale='Viridis',
    colorbar_title='IV (%)'
)])

fig.update_layout(
    title=f'{symbol} Implied Volatility Surface',
    scene=dict(
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Days to Expiry',
        zaxis_title='IV (%)'
    ),
    template='plotly_dark',
    height=600
)
fig.show()


## 7. Put-Call Parity Arbitrage Check

In [ ]:
# Merge calls and puts by strike
calls_parity = calls[['strike', 'mid']].rename(columns={'mid': 'call_price'})
puts_parity = puts[['strike', 'mid']].rename(columns={'mid': 'put_price'})

parity = calls_parity.merge(puts_parity, on='strike')

# Put-Call Parity: C - P = S - K*e^(-rT)
r = 0.05
T = 30 / 365

parity['theoretical_diff'] = spot - parity['strike'] * np.exp(-r * T)
parity['actual_diff'] = parity['call_price'] - parity['put_price']
parity['arbitrage'] = parity['actual_diff'] - parity['theoretical_diff']
parity['arbitrage_pct'] = (parity['arbitrage'] / spot) * 100

print("Put-Call Parity Analysis:")
print(parity[['strike', 'theoretical_diff', 'actual_diff', 'arbitrage', 'arbitrage_pct']].round(4))

# Flag significant arbitrage
arbitrage_threshold = 0.5  # 50 cents
arb_opportunities = parity[abs(parity['arbitrage']) > arbitrage_threshold]
print(f"\nPotential arbitrage opportunities: {len(arb_opportunities)}")


## 8. Portfolio Greeks Aggregation

In [ ]:
# Simulate a portfolio of options
portfolio = [
    {'strike': spot * 1.0, 'type': 'call', 'quantity': 10, 'position': 'long'},   # ATM call
    {'strike': spot * 0.95, 'type': 'put', 'quantity': 5, 'position': 'long'},    # OTM put hedge
    {'strike': spot * 1.05, 'type': 'call', 'quantity': -5, 'position': 'short'}, # Covered call
]

# Calculate portfolio Greeks
T = 30 / 365
r = 0.05

port_greeks = {'delta': 0, 'gamma': 0, 'vega': 0, 'theta': 0}

for pos in portfolio:
    K = pos['strike']
    iv = 0.20  # simplified
    qty = pos['quantity'] if pos['position'] == 'long' else -pos['quantity']
    
    port_greeks['delta'] += qty * 100 * bs_delta(spot, K, T, r, iv, pos['type'])
    port_greeks['gamma'] += qty * 100 * bs_gamma(spot, K, T, r, iv)
    port_greeks['vega'] += qty * 100 * bs_vega(spot, K, T, r, iv)
    port_greeks['theta'] += qty * 100 * bs_theta(spot, K, T, r, iv, pos['type'])

print("\nPortfolio Greeks:")
print(f"  Delta (Δ): {port_greeks['delta']:+.1f} shares equivalent")
print(f"  Gamma (Γ): {port_greeks['gamma']:.2f} delta/$ move")
print(f"  Vega (ν):  ${port_greeks['vega']:.0f} per 1% vol")
print(f"  Theta (θ): ${port_greeks['theta']:.0f} per day")

print(f"\nInterpretation:")
print(f"  - Position is {'BULLISH' if port_greeks['delta'] > 0 else 'BEARISH'} (Delta)")
print(f"  - {'LONG' if port_greeks['gamma'] > 0 else 'SHORT'} gamma (convexity)")
print(f"  - {'LONG' if port_greeks['vega'] > 0 else 'SHORT'} volatility")


## 9. Summary Report

In [ ]:
print("="*70)
print("VOLATILITY SURFACE & OPTIONS GREEKS ANALYSIS")
print("="*70)

print(f"\n1. UNDERLYING")
print(f"   Symbol: {symbol}")
print(f"   Spot: ${spot:.2f}")

print(f"\n2. VOLATILITY METRICS")
print(f"   ATM IV: {atm_iv*100:.1f}%" if not np.isnan(atm_iv) else "   ATM IV: N/A")
print(f"   Skew: {skew:.1f}%" if not np.isnan(skew) else "   Skew: N/A")
if vix_hist_df is not None:
    print(f"   VIX (current): {vix_hist_df['vix'].iloc[-1]:.1f}")
    print(f"   VIX (avg): {vix_hist_df['vix'].mean():.1f}")

print(f"\n3. GREEKS SUMMARY")
print(f"   Peak Gamma strike: ${calls.loc[calls['gamma'].idxmax(), 'strike']:.0f}")
print(f"   Peak Vega strike: ${calls.loc[calls['vega'].idxmax(), 'strike']:.0f}")

print(f"\n4. ARBITRAGE CHECK")
print(f"   Put-Call parity violations: {len(arb_opportunities)}")
print(f"   Max deviation: ${parity['arbitrage'].abs().max():.3f}")

print("\n" + "="*70)


## Summary

This CQF-level example covered:

1. **VIX Analysis**: Term structure, historical regimes
2. **Options Chains**: Live/synthetic data handling
3. **Volatility Smile**: IV extraction, skew measurement
4. **Greeks Calculation**: Delta, Gamma, Vega, Theta
5. **3D Vol Surface**: Strike x Expiry x IV
6. **Arbitrage Detection**: Put-Call parity checks
7. **Portfolio Aggregation**: Combined Greeks exposure

**Applications**:
- Options market making
- Volatility trading strategies
- Risk management
- Derivatives pricing